# P7 SGN clustering

Set the paths marked in the code before running this notebook.


In [ ]:
#1. =========== Load counts and set the random seed
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
import os
import random
import tarfile

for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[key] = "1"

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.io import mmread
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from threadpoolctl import threadpool_limits

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
thread_guard = threadpool_limits(limits=1)
sc.settings.n_jobs, sc.settings.verbosity = 1, 0
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 10,
    "figure.dpi": 120, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
})

ROOT = Path(".")  # Set the project root before running this template.
# Set the original count-data file or directory path.
if not (ROOT).is_file():
    ROOT = ROOT.parent
# Set the original count-data file or directory path.
SOURCE = ROOT

with tarfile.open(SOURCE, "r:gz") as archive:
    features = pd.read_csv(archive.extractfile("SET7_matrix/genes.tsv"), sep="\t", header=None)
    barcodes = pd.read_csv(archive.extractfile("SET7_matrix/barcodes.tsv"), sep="\t", header=None)[0].astype(str)
    matrix = mmread(archive.extractfile("SET7_matrix/matrix.mtx")).T.tocsr()

assert matrix.shape == (len(barcodes), len(features))
symbols = features.iloc[:, 1].astype(str) if features.shape[1] > 1 else features.iloc[:, 0].astype(str)
gene_ids = features.iloc[:, 0].astype(str)
codes, unique = pd.factorize(symbols, sort=False)
mapping = sparse.csr_matrix(
    (np.ones(len(codes), dtype=np.int32), (np.arange(len(codes)), codes)),
    shape=(len(codes), len(unique)),
)
X = (matrix @ mapping).astype(np.int32).tocsr()
var = pd.DataFrame(index=pd.Index(unique, name="gene_symbol"))
var["source_gene_ids"] = pd.Series(gene_ids.to_numpy()).groupby(codes).agg(";".join).to_numpy()
var["mt"] = var.index.str.match(r"(?i)^mt[-_]")
var["hb"] = var.index.str.match(r"(?i)^hb[ab]-")
var["ribo"] = var.index.str.match(r"^Rp[sl]\d")
var["technical"] = var.index.str.match(r"(?i)^ercc[-_]\d+$|^(e?gfp\d*|tdtomato|mcherry|wpre)$")
obs = pd.DataFrame(index=pd.Index(
    [f"Local_P7_SET7__SET7__{x}" for x in barcodes], name="cell_id_global"
))
obs["original_cell_id"] = barcodes.to_numpy()
obs["dataset_id"], obs["library_id"] = "Local_P7_SET7", "SET7"
obs["stage_standardized"] = "P7"
a = ad.AnnData(X=X, obs=obs, var=var)
a.layers["counts"] = X.copy()

In [ ]:
#2. =========== Quality control
X = a.layers["counts"]
total = np.asarray(X.sum(axis=1)).ravel()
a.obs["total_counts"] = total
a.obs["n_genes"] = X.getnnz(axis=1)
for feature in ("mt", "hb", "ribo", "technical"):
    a.obs[f"pct_{feature}"] = (
        100 * np.asarray(X[:, a.var[feature].to_numpy()].sum(axis=1)).ravel()
        / np.maximum(total, 1)
    )
a.obs["endogenous_counts"] = np.asarray(
    X[:, ~a.var["technical"].to_numpy()].sum(axis=1)
).ravel()
MT_MAX_PCT = 12.5
a.obs["mito_or_zero_fail"] = (
    (a.obs.pct_mt > MT_MAX_PCT)
    | (a.obs.endogenous_counts == 0)
    | (a.obs.n_genes == 0)
)
post_mito = a[~a.obs.mito_or_zero_fail.to_numpy()].copy()

fig, axes = plt.subplots(1, 4, figsize=(10, 2.7))
qc_panels = [
    ("total_counts", "log10(UMI + 1)", lambda x: np.log10(x + 1)),
    ("n_genes", "Detected genes", lambda x: x),
    ("pct_mt", "Mitochondrial %", lambda x: x),
    ("pct_hb", "log10(Hb % + 1)", lambda x: np.log10(x + 1)),
]
for ax, (column, title, transform) in zip(axes, qc_panels):
    values = [
        transform(a.obs[column].to_numpy()),
        transform(post_mito.obs[column].to_numpy()),
    ]
    ax.violinplot(values, showmedians=True, showextrema=False)
    ax.set(xticks=[1, 2], xticklabels=["Input", "Retained"], title=title)
    if column == "pct_mt":
        ax.axhline(MT_MAX_PCT, color="#AE814C", linestyle="--", linewidth=1)
fig.tight_layout()
plt.show()

In [ ]:
#3. =========== Global HVG2000 clustering
GLOBAL_PCS, GLOBAL_NEIGHBORS = 40, 25
GLOBAL_RESOLUTION = 0.6
analysis_genes = ~(post_mito.var["mt"] | post_mito.var["technical"]).to_numpy()
global_expression = post_mito[:, analysis_genes].copy()
global_expression.X = global_expression.layers["counts"].astype(np.float32).copy()
sc.pp.normalize_total(global_expression, target_sum=10000)
sc.pp.log1p(global_expression)

hvg_eligible = (
    (global_expression.layers["counts"].getnnz(axis=0) >= 5)
    & ~global_expression.var["hb"].to_numpy()
)
hvg_input = global_expression[:, hvg_eligible].copy()
sc.pp.highly_variable_genes(
    hvg_input, n_top_genes=5000, flavor="seurat_v3", layer="counts", span=0.3,
)
hvg_ranked = hvg_input.var["highly_variable_rank"].nsmallest(5000).index
genes = hvg_ranked[:2000].tolist()
branch = global_expression[:, genes].copy()
branch.X = branch.X.toarray().astype(np.float32)
sc.pp.scale(branch, max_value=10)
sc.tl.pca(branch, n_comps=GLOBAL_PCS, svd_solver="arpack", random_state=RANDOM_SEED)
sc.pp.neighbors(
    branch, n_neighbors=GLOBAL_NEIGHBORS, n_pcs=GLOBAL_PCS,
    transformer="sklearn", random_state=RANDOM_SEED,
)
sc.tl.leiden(
    branch, resolution=GLOBAL_RESOLUTION, flavor="igraph",
    directed=False, n_iterations=2, random_state=RANDOM_SEED, key_added="cluster",
)
global_expression.obs["cluster_HVG2000"] = branch.obs["cluster"].copy()

In [ ]:
#4. =========== SGN-enriched local HVG2000 clustering
candidate_mask = global_expression.obs["cluster_HVG2000"].astype(str).isin(("4", "9")).to_numpy()
sgn_candidate = post_mito[candidate_mask].copy()
sgn_candidate.obs["global_cluster_HVG2000"] = (
    global_expression.obs.loc[sgn_candidate.obs_names, "cluster_HVG2000"].astype(str).to_numpy()
)
analysis_gene_mask = ~(sgn_candidate.var["mt"] | sgn_candidate.var["technical"]).to_numpy()
sgn_expression = sgn_candidate[:, analysis_gene_mask].copy()
sgn_expression.X = sgn_expression.layers["counts"].astype(np.float32).copy()
sc.pp.normalize_total(sgn_expression, target_sum=10000)
sc.pp.log1p(sgn_expression)

hvg_eligible = (
    (sgn_expression.layers["counts"].getnnz(axis=0) >= 5)
    & ~sgn_expression.var["hb"].to_numpy()
)
hvg_input = sgn_expression[:, hvg_eligible].copy()
sc.pp.highly_variable_genes(
    hvg_input, n_top_genes=5000, flavor="seurat_v3", layer="counts", span=0.3,
)
sub_hvg_ranked = hvg_input.var["highly_variable_rank"].nsmallest(5000).index.tolist()
branch = sgn_expression[:, sub_hvg_ranked[:2000]].copy()
branch.X = branch.X.toarray().astype(np.float32)
sc.pp.scale(branch, max_value=10)
sc.tl.pca(branch, n_comps=30, svd_solver="arpack", random_state=RANDOM_SEED)
sc.pp.neighbors(
    branch, n_neighbors=20, n_pcs=30,
    transformer="sklearn", random_state=RANDOM_SEED,
)
sc.tl.leiden(
    branch, resolution=0.5, flavor="igraph", directed=False,
    n_iterations=2, random_state=RANDOM_SEED, key_added="cluster",
)
sgn_expression.obs["local_HVG2000"] = branch.obs["cluster"].copy()

In [ ]:
#5. =========== Refined neuronal candidates
keep = sgn_expression.obs["local_HVG2000"].astype(str).isin(("0", "2", "3"))
sgn_refined_counts = sgn_candidate[keep.to_numpy()].copy()
sgn_refined_counts.obs["source_local_cluster"] = (
    sgn_expression.obs.loc[sgn_refined_counts.obs_names, "local_HVG2000"].astype(str).to_numpy()
)
assert sgn_refined_counts.n_obs == 998
analysis_mask = ~(sgn_refined_counts.var["mt"] | sgn_refined_counts.var["technical"]).to_numpy()
sgn_refined_expression = sgn_refined_counts[:, analysis_mask].copy()
sgn_refined_expression.X = sgn_refined_expression.layers["counts"].astype(np.float32).copy()
sc.pp.normalize_total(sgn_refined_expression, target_sum=10000)
sc.pp.log1p(sgn_refined_expression)

In [ ]:
#6. =========== Marker-only clustering
core_groups = {
    "Ia-like": ["Calb2", "Pcdh20", "Rxrg"],
    "Ib-like": ["Calb1", "Ttn"],
    "Ic-like": ["Lypd1", "Pou4f1", "Grm8"],
    "Type II": ["Ngfr", "Prph", "Plk5"],
}
core_genes = [gene for group in core_groups.values() for gene in group]
raw_core = sgn_refined_expression[:, core_genes].layers["counts"]
core_detected_n = raw_core.getnnz(axis=1)
informative = core_detected_n > 0
p7_marker_only = sgn_refined_expression[informative, core_genes].copy()
p7_marker_only.X = p7_marker_only.X.toarray().astype(np.float32)
p7_marker_only.obs["core_markers_detected"] = core_detected_n[informative]
sc.pp.neighbors(
    p7_marker_only, n_neighbors=15, use_rep="X", metric="euclidean",
    transformer="sklearn", random_state=RANDOM_SEED,
)
sc.tl.leiden(
    p7_marker_only, resolution=0.5, flavor="igraph", directed=False,
    n_iterations=2, random_state=RANDOM_SEED, key_added="p7_marker_cluster",
)
sc.tl.umap(p7_marker_only, min_dist=0.35, random_state=RANDOM_SEED)

In [ ]:
#7. =========== Final marker panel
cluster = p7_marker_only.obs["p7_marker_cluster"].astype(str)
type_map = {
    "0": "Ia-like", "4": "Ia-like",
    "9": "Ib-like", "13": "Ib-like",
    "11": "Ic-like", "3": "Type II",
}
type_order = ["Ia-like", "Ib-like", "Ic-like", "Type II"]
workbook_groups = {
    "Ia markers": ["Calb2", "Pcdh20", "Rxrg", "Cxcl14"],
    "Ib markers": ["Calb1", "Ttn", "Runx1"],
    "Ic markers": ["Lypd1", "Pou4f1", "Grm8", "Ntng1"],
    "II markers": ["Ngfr", "Prph", "Plk5", "Th", "Etv4", "Gabrg2"],
}
workbook_genes = [gene for group in workbook_groups.values() for gene in group]
de_obj = sgn_refined_expression[p7_marker_only.obs_names, :].copy()
eligible = de_obj.layers["counts"].getnnz(axis=0) >= 10
for flag in ("mt", "hb", "ribo", "technical"):
    eligible &= ~de_obj.var[flag].to_numpy()
de_obj = de_obj[:, eligible].copy()
provisional_type = cluster.map(type_map).fillna("Other")
de_obj.obs["P7_provisional_type"] = pd.Categorical(
    provisional_type.to_numpy(), categories=type_order + ["Other"], ordered=True,
)
sc.tl.rank_genes_groups(
    de_obj, groupby="P7_provisional_type", groups=type_order,
    reference="rest", method="wilcoxon", corr_method="benjamini-hochberg",
    n_genes=min(300, de_obj.n_vars), use_raw=False, key_added="P7_provisional_DE",
)
lineage_genes = [
    gene for gene in ["Snap25", "Nefl", "Isl1", "Sox10", "Col2a1", "Ptprc"]
    if gene in sgn_refined_expression.var_names
]
excluded = set(workbook_genes + lineage_genes)
extra_markers = {group: [] for group in type_order}
for group in type_order:
    ranked = sc.get.rank_genes_groups_df(
        de_obj, group=group, key="P7_provisional_DE"
    ).copy()
    genes = ranked["names"].astype(str).tolist()
    raw = de_obj[:, genes].layers["counts"]
    in_group = de_obj.obs["P7_provisional_type"].astype(str).eq(group).to_numpy()
    ranked["pct_in"] = raw[in_group].getnnz(axis=0) / in_group.sum()
    ranked["pct_rest"] = raw[~in_group].getnnz(axis=0) / (~in_group).sum()
    ranked["pct_difference"] = ranked["pct_in"] - ranked["pct_rest"]
    candidates = ranked.loc[
        ~ranked["names"].isin(excluded)
        & ranked["pvals_adj"].lt(0.05)
        & ranked["logfoldchanges"].gt(0.5)
        & ranked["pct_in"].ge(0.25)
        & ranked["pct_difference"].ge(0.15)
    ].sort_values(
        ["pvals_adj", "pct_difference"], ascending=[True, False]
    ).head(2)
    extra_markers[group] = candidates["names"].astype(str).tolist()
    excluded.update(extra_markers[group])

plot_groups = dict(workbook_groups)
for group, title in [
    ("Ia-like", "DE Ia"), ("Ib-like", "DE Ib"),
    ("Ic-like", "DE Ic"), ("Type II", "DE II"),
]:
    if extra_markers[group]:
        plot_groups[title] = extra_markers[group]
plot_groups["Lineage check"] = lineage_genes
plot_genes = [gene for group in plot_groups.values() for gene in group]
purple = LinearSegmentedColormap.from_list(
    "P7_expanded_purple",
    ["#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F"],
)

In [ ]:
#8. =========== Final P7 four-type UMAP and DotPlot

final_map = {
    "0": "Type Ia", "4": "Type Ia",
    "9": "Type Ib", "13": "Type Ib",
    "11": "Type Ic",
    "3": "Type II",
}
final_order = ["Type Ia", "Type Ib", "Type Ic", "Type II"]
final_colors = {
    "Type Ia": "#557B9F",
    "Type Ib": "#B7864A",
    "Type Ic": "#9A6E9C",
    "Type II": "#6E9368",
}

source_clusters = p7_marker_only.obs["p7_marker_cluster"].astype(str)
selected = source_clusters.isin(final_map)
final_ids = p7_marker_only.obs_names[selected.to_numpy()]

p7_final = sgn_refined_expression[final_ids, :].copy()
p7_final.obs["source_marker_cluster"] = source_clusters.loc[final_ids].to_numpy()
p7_final.obs["P7_final_type"] = pd.Categorical(
    source_clusters.loc[final_ids].map(final_map).to_numpy(),
    categories=final_order,
    ordered=True,
)
p7_final.obsm["X_umap"] = p7_marker_only.obsm["X_umap"][
    selected.to_numpy()
].copy()
assert p7_final.n_obs == 387


# Final UMAP: legend and label versions use the same coordinates and colors.
xy = p7_final.obsm["X_umap"]
labels = p7_final.obs["P7_final_type"].astype(str).to_numpy()
margin = 0.07 * np.ptp(xy, axis=0)
lower, upper = xy.min(axis=0) - margin, xy.max(axis=0) + margin

for mode in ("legend", "label"):
    fig, ax = plt.subplots(
        figsize=(6.5, 4.5) if mode == "legend" else (4.9, 4.5)
    )
    fig.subplots_adjust(
        left=0.12 if mode == "legend" else 0.14,
        right=0.68 if mode == "legend" else 0.96,
        bottom=0.12, top=0.91,
    )

    for group in final_order:
        mask = labels == group
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            s=1.35, alpha=0.82, linewidths=0,
            color=final_colors[group],
        )

    ax.set(
        xlim=(lower[0], upper[0]),
        ylim=(lower[1], upper[1]),
        xlabel="UMAP1", ylabel="UMAP2",
        title="P7 SGN / four final types",
    )
    ax.set_box_aspect(1)

    if mode == "legend":
        handles = [
            Line2D(
                [], [], marker="o", linestyle="",
                color=final_colors[group], markersize=5.2,
                label=f"{group} ({(labels == group).sum()})",
            )
            for group in final_order
        ]
        ax.legend(
            handles=handles, loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False, fontsize=7.5,
        )
    else:
        for group in final_order:
            points = xy[labels == group]
            center = np.median(points, axis=0)
            anchor = points[np.argmin(((points - center) ** 2).sum(axis=1))]
            ax.text(
                anchor[0], anchor[1], group,
                ha="center", va="center", fontsize=8,
                path_effects=[
                    pe.withStroke(linewidth=1.6, foreground="white")
                ],
            )
    plt.show()

# Final DotPlot: recompute both metrics from the 387 selected cells.
final_plot = p7_final[:, plot_genes].copy()
raw = final_plot.layers["counts"]
detected = (raw.toarray() if sparse.issparse(raw) else np.asarray(raw)) > 0
log_expr = (
    final_plot.X.toarray()
    if sparse.issparse(final_plot.X)
    else np.asarray(final_plot.X)
)

final_pct = pd.DataFrame(
    detected, index=final_plot.obs_names, columns=plot_genes
).groupby(
    final_plot.obs["P7_final_type"], observed=True
).mean().reindex(final_order)
final_avg = pd.DataFrame(
    log_expr, index=final_plot.obs_names, columns=plot_genes
).groupby(
    final_plot.obs["P7_final_type"], observed=True
).mean().reindex(final_order)
final_scaled_avg = final_avg.sub(
    final_avg.mean(axis=0), axis=1
).div(
    final_avg.std(axis=0, ddof=0).replace(0, 1), axis=1
)

dp = sc.pl.DotPlot(
    final_plot,
    var_names=plot_groups,
    groupby="P7_final_type",
    use_raw=False,
    dot_color_df=final_scaled_avg,
    dot_size_df=final_pct,
    categories_order=final_order,
    vmin=-1, vmax=1,
    figsize=(17.5, 4.2),
)
dp.style(
    cmap=purple, dot_min=0, dot_max=1,
    smallest_dot=0, largest_dot=55,
    size_exponent=1, dot_edge_lw=0, grid=False,
)
dp.legend(
    size_title="% Exp.",
    colorbar_title="Avg. Exp. (gene z-score)",
)
dp.show()

In [ ]:
#9. =========== Save P7 integration H5AD

from datetime import datetime
from importlib.metadata import version
from pathlib import Path
import hashlib
import os
import tempfile

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

ad.settings.allow_write_nullable_strings = True

# Set the output directory path.
output_dir = ROOT
# Set the result file path; uses P7_Local_P7_SET7__SET7__SGN_for_integration.h5ad generated by P7 clustering export.
old_path = ROOT
# Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
output_path = ROOT
# Set the data file path; uses P7_Local_P7_SET7__SET7.h5ad generated by source quality control.
qc_path = ROOT
# Set the original count-data file or directory path.
raw_path = ROOT
input_path = old_path if old_path.exists() else output_path

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def ids_sha256(values):
    encoded = "\n".join(map(str, values)).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def csr_sha256(matrix):
    matrix = matrix.tocsr()
    digest = hashlib.sha256()
    digest.update(np.asarray(matrix.shape, dtype="<i8").tobytes())
    digest.update(np.asarray(matrix.indptr, dtype="<i8").tobytes())
    digest.update(np.asarray(matrix.indices, dtype="<i8").tobytes())
    digest.update(np.asarray(matrix.data, dtype="<i4").tobytes())
    return digest.hexdigest()

selected_ids = p7_final.obs_names
labels = p7_final.obs["P7_final_type"].astype(str)
label_order = ["Type Ia", "Type Ib", "Type Ic", "Type II"]

if input_path.exists():
    previous = ad.read_h5ad(input_path)
else:
    # A fresh export starts from the QC parent, not a previous export.
    qc = ad.read_h5ad(qc_path)
    excluded_flags = ["mt_feature", "hb_feature", "ERCC_feature",
                      "reporter_or_vector_feature", "technical_feature", "external_spike_feature"]
    gene_mask = (
        qc.var["eligible_for_within_dataset_analysis"].to_numpy(dtype=bool)
        & ~qc.var[excluded_flags].any(axis=1).to_numpy(dtype=bool)
        & (qc.X.getnnz(axis=0) > 0)
    )
    previous = qc[selected_ids, gene_mask].copy()
    previous.X = previous.X.tocsr().astype(np.int32)
    previous.layers["counts"] = previous.X.copy()
    previous.uns.clear()
    batch = previous.obs["dataset_id"].astype(str) + "__" + previous.obs["library_id"].astype(str)
    previous.obs["sample_id"] = batch.to_numpy()
    previous.obs["integration_batch"] = batch.to_numpy()
    previous.obs["source_marker_cluster"] = p7_final.obs["source_marker_cluster"].astype(str).to_numpy()
    previous.obs["doublet_score"] = post_mito.obs.loc[selected_ids, "doublet_score"].to_numpy()
    previous.obs["blood_qc_pass"] = True
    program_signal = np.zeros(previous.n_obs, dtype=bool)
    for panel in (["Ptprc", "Lyz2", "Tyrobp", "C1qa", "C1qb", "Lcp1"],
                  ["Gypa", "Klf1", "Alas2", "Slc4a1"]):
        present = [g for g in panel if g in qc.var_names]
        program_signal |= np.asarray((qc[selected_ids, present].X > 0).sum(axis=1)).ravel() >= 2
    previous.obs["blood_review_signal"] = program_signal
    previous.var["integration_gene_id_candidate"] = previous.var["canonical_gene_id"].copy()
    input_path = qc_path
    del qc

assert selected_ids.is_unique
assert previous.obs_names.equals(selected_ids)
assert set(labels) == set(label_order)
assert "counts" in previous.layers
assert sparse.isspmatrix_csr(previous.X)
assert sparse.isspmatrix_csr(previous.layers["counts"])
assert (previous.X != previous.layers["counts"]).nnz == 0

previous_total = int(previous.layers["counts"].sum())
previous_nnz = int(previous.layers["counts"].nnz)
excluded_ids = []

if "tdtomato2" in previous.var_names:
    reporter_counts = previous[:, "tdtomato2"].layers["counts"]
    reporter_x = previous[:, "tdtomato2"].X
    assert reporter_counts.nnz == 0 and reporter_x.nnz == 0
    excluded_ids.append("tdtomato2")

keep = ~previous.var_names.isin(excluded_ids)
adata = previous[:, keep].copy()
adata.X = adata.layers["counts"].copy()
adata.raw = None
adata.uns.clear()
adata.obsm.clear()
adata.obsp.clear()
adata.varm.clear()
adata.varp.clear()

counts = adata.layers["counts"]
assert sparse.isspmatrix_csr(counts)
assert counts.dtype == np.int32
assert np.isfinite(counts.data).all()
assert (counts.data > 0).all()
assert int(counts.sum()) == previous_total
assert int(counts.nnz) == previous_nnz
assert adata.var_names.is_unique
assert len({gene.casefold() for gene in adata.var_names}) == adata.n_vars

qc = ad.read_h5ad(qc_path)
assert adata.obs_names.isin(qc.obs_names).all()
assert adata.var_names.isin(qc.var_names).all()
assert (counts != qc[adata.obs_names, adata.var_names].X).nnz == 0
assert adata.var_names.isin(p7_final.var_names).all()
assert (counts != p7_final[:, adata.var_names].layers["counts"]).nnz == 0

if "legacy_sample_id" not in adata.obs:
    adata.obs["legacy_sample_id"] = adata.obs["sample_id"].astype(str)
if "source_qc_rule_set" not in adata.obs:
    adata.obs["source_qc_rule_set"] = adata.obs["qc_rule_set"].astype(str)
if "source_qc_evidence_scope" not in adata.obs:
    adata.obs["source_qc_evidence_scope"] = adata.obs["qc_evidence_scope"].astype(str)

adata.obs["cell_id_global"] = adata.obs_names.astype(str)
adata.obs["sample_id"] = pd.Series(
    pd.NA, index=adata.obs_names, dtype="string"
)
adata.obs["batch_id_global"] = adata.obs["integration_batch"].astype(str).to_numpy()
adata.obs["batch_stage_confounded"] = True
adata.obs["stage_as_batch"] = False
adata.obs["stage_analysis"] = "P7"
adata.obs["stage_standardized"] = "P7"
adata.obs["matrix_unit"] = "UMI_counts"
adata.obs["matrix_unit_verification"] = "user_confirmed"
adata.obs["raw_counts_available"] = True
adata.obs["raw_counts_status"] = "available_verified"

adata.obs["P7_final_type"] = pd.Categorical(
    labels.to_numpy(), categories=label_order, ordered=True
)
adata.obs["cell_type"] = pd.Categorical(
    labels.to_numpy(), categories=label_order, ordered=True
)
adata.obs["scanvi_label"] = pd.Categorical(
    labels.to_numpy(), categories=label_order, ordered=True
)
adata.obs["selected_for_integration"] = True
adata.obs["annotation_status"] = "final_stagewise_annotation"
adata.obs["annotation_source"] = "P7_final_type_manual_marker_review"
adata.obs["scanvi_supervision_status"] = "labeled_final_stagewise_annotation"
adata.obs["cell_scope_status"] = "final_P7_four_type_selection"
adata.obs["lineage_status"] = "SGN_final_four_types"
adata.obs["qc_rule_set"] = "source_technical_QC_then_P7_mito12p5_or_zero"
adata.obs["qc_evidence_scope"] = "verified_UMI_counts_and_P7_stagewise_QC"

adata.obs["doublet_model_run"] = True
adata.obs["doublet_status"] = "scrublet_score_only"
adata.obs["predicted_doublet"] = pd.array(
    [pd.NA] * adata.n_obs, dtype="boolean"
)
adata.obs["doublet_qc_pass"] = pd.array(
    [pd.NA] * adata.n_obs, dtype="boolean"
)
adata.obs["total_counts_analysis"] = np.asarray(counts.sum(axis=1)).ravel()
adata.obs["n_genes_analysis"] = counts.getnnz(axis=1)

if "legacy_feature_row_id" not in adata.var:
    adata.var["legacy_feature_row_id"] = adata.var["feature_row_id"].astype(str)
if "source_mapping_status" not in adata.var:
    adata.var["source_mapping_status"] = (
        adata.var["canonical_mapping_status"].astype(str)
    )
if "source_ribosomal_feature" not in adata.var:
    adata.var["source_ribosomal_feature"] = (
        adata.var["ribosomal_feature"].astype(bool)
    )

candidate = adata.var["integration_gene_id_candidate"].astype(str)
old_canonical = adata.var["canonical_gene_id"]
known_canonical = old_canonical.notna()
assert (
    old_canonical.loc[known_canonical].astype(str).to_numpy()
    == candidate.loc[known_canonical].to_numpy()
).all()

source_rows = pd.Series(
    np.arange(1, len(features) + 1, dtype=np.int64),
    index=features.iloc[:, 0].astype(str),
)
adata.var["feature_row_id"] = (
    adata.var["source_feature_id"].astype(str).map(source_rows)
)
assert adata.var["feature_row_id"].notna().all()
adata.var["feature_row_id"] = adata.var["feature_row_id"].astype(np.int64)

source_ids = adata.var["gene_id_original"].astype(str)
is_ensembl = source_ids.str.fullmatch(r"ENSMUSG\d+(?:\.\d+)?")
adata.var["source_gene_id_type"] = np.where(
    source_ids.str.contains(";", regex=False),
    "ensembl_gene_id_list",
    np.where(is_ensembl, "ensembl_gene_id", "unresolved"),
)
adata.var["gene_symbol"] = adata.var_names.astype(str)
adata.var["gene_id_analysis"] = adata.var_names.astype(str)
adata.var["gene_id_type"] = "gene_symbol"
adata.var["feature_type"] = "Gene Expression"
adata.var["integration_gene_id_candidate_type"] = np.where(
    candidate.str.fullmatch(r"ENSMUSG\d+(?:\.\d+)?"),
    "ensembl_gene_id",
    "unresolved",
)
adata.var["canonical_gene_id"] = pd.Series(
    pd.NA, index=adata.var_names, dtype="string"
)
adata.var["canonical_mapping_status"] = "not_run"
adata.var["eligible_for_cross_dataset_integration"] = False
adata.var["cross_dataset_feature_blocker"] = (
    "versioned_cross_dataset_gene_harmonization_not_run"
)
adata.var["hemoglobin_feature"] = adata.var["hb_feature"].astype(bool)
adata.var["ribosomal_feature"] = pd.array(
    [pd.NA] * adata.n_vars, dtype="boolean"
)
adata.var["use_for_model_input"] = True
adata.var["n_cells_in_export"] = counts.getnnz(axis=0)
adata.var["total_counts_in_export"] = np.asarray(counts.sum(axis=0)).ravel()
adata.var["detected_in_exported_cells"] = (
    adata.var["total_counts_in_export"] > 0
)

previous_provenance = previous.uns.get("source_provenance", {})
migration_history = previous_provenance.get("migration_history", {
    "previous_matrix_contract": str(previous.uns.get("matrix_contract", {})),
    "previous_integration_contract": str(previous.uns.get("integration_contract", {})),
    "previous_sample_id": str(previous.obs["sample_id"].iloc[0]),
    "previous_feature_row_id_semantics": "gene_symbol",
})

cell_hash = ids_sha256(adata.obs_names)
feature_hash = ids_sha256(adata.var_names)
x_hash = csr_sha256(adata.X)
counts_hash = csr_sha256(counts)
label_counts = {
    name: int((labels == name).sum()) for name in label_order
}

adata.uns["matrix_contract"] = {
    "schema_version": "2.0.0",
    "raw_counts_available": True,
    "raw_counts_status": "available_verified",
    "counts_layer": "counts",
    "counts_layer_semantics": "raw_counts",
    "count_type": "UMI",
    "counts_verification_source": (
        "user_confirmed_UMI; exact_match_to_P7_QC_and_notebook_counts"
    ),
    "counts_dtype": str(counts.dtype),
    "counts_sparse_format": "CSR",
    "X_semantics": "raw_integer_UMI_counts",
    "X_dtype": str(adata.X.dtype),
    "X_sparse_format": "CSR",
    "X_normalization_status": "not_applied",
    "X_log1p_status": "not_applied",
    "X_scaling_status": "not_applied",
    "X_counts_relationship": "exact_copy",
    "raw_present": False,
}
adata.uns["selection_contract"] = {
    "source_notebook": 'fig1_P7_clustering.ipynb',
    "source_object": "p7_final",
    "selection_rule": (
        "global_HVG2000_4,9 -> local_HVG2000_0,2,3 -> "
        "core_marker_positive -> marker_clusters_0,4,9,13,11,3"
    ),
    "source_cluster_to_type": dict(final_map),
    "n_selected_cells": adata.n_obs,
    "cell_type_counts": label_counts,
    "annotation_status": "final_stagewise_annotation",
    "annotation_source": "P7_final_type_manual_marker_review",
    "label_policy": "all_final_cell_types",
    "blood_review_signal_cells": int(adata.obs["blood_review_signal"].sum()),
}
adata.uns["integration_contract"] = {
    "schema_name": "auditory_neuron_stagewise",
    "schema_version": "2.0.0",
    "input_role": "stagewise_pre_harmonization",
    "counts_layer": "counts",
    "batch_key": "integration_batch",
    "labels_key": "scanvi_label",
    "unlabeled_category": "Unknown",
    "label_policy": "all_final_cell_types",
    "gene_harmonization_status": "not_run",
    "cross_dataset_blocker": (
        "versioned_cross_dataset_gene_harmonization_not_run"
    ),
    "batch_stage_confounded": True,
    "count_model_compatible": True,
    "model_ready": False,
    "model_blockers": [
        "versioned_cross_dataset_gene_harmonization_not_run",
        "P7_binary_doublet_call_not_completed",
    ],
}
# Set the original count-data file or directory path.
# Set the data file path; uses P7_Local_P7_SET7__SET7.h5ad generated by source quality control.
adata.uns["source_provenance"] = {
    "schema_version": "2.0.0",
    "source_files": {
        "raw_matrix": str(ROOT),
        "QC_mother": str(ROOT),
        "migration_input": str(input_path.relative_to(ROOT)),
    },
    "source_file_sha256": {
        "raw_matrix": file_sha256(raw_path),
        "QC_mother": file_sha256(qc_path),
        "migration_input": file_sha256(input_path),
    },
    "source_notebook": 'fig1_P7_clustering.ipynb',
    "source_dataset": "Local_P7_SET7",
    "source_library": "SET7",
    "source_object": "p7_final",
    "source_reference_assembly": "unknown",
    "source_annotation_release": "unknown",
    "source_reference_evidence": "not_verified",
    "software_versions": {
        name: version(name)
        for name in ("anndata", "numpy", "pandas", "scipy", "scanpy")
    },
    "export_date": datetime.now().astimezone().isoformat(timespec="seconds"),
    "qc_metric_definitions": (
        "Source QC percentages are 0-100 and retain their original "
        "pre-feature-exclusion definitions; analysis counts use the "
        "exported counts layer."
    ),
    "migration_history": migration_history,
}
adata.uns["export_contract"] = {
    "output_filename": output_path.name,
    "n_obs": adata.n_obs,
    "n_vars": adata.n_vars,
    "saved_slots": ["X", "obs", "var", "layers/counts", "uns"],
    "excluded_feature_classes": [
        "mitochondrial", "hemoglobin", "ERCC", "reporter_or_vector",
        "technical",
    ],
    "excluded_feature_ids": excluded_ids,
    "ordered_cell_id_sha256": cell_hash,
    "ordered_feature_id_sha256": feature_hash,
    "X_sha256": x_hash,
    "counts_sha256": counts_hash,
    "X_nnz": int(adata.X.nnz),
    "counts_nnz": int(counts.nnz),
    "total_counts": int(counts.sum()),
    "fingerprint_encoding": (
        "SHA256; IDs UTF-8 joined by LF without trailing LF; "
        "CSR shape/indptr/indices little-endian int64; "
        "CSR data little-endian int32"
    ),
}

output_dir.mkdir(parents=True, exist_ok=True)

# Set the temporary working directory or file path.
temp_dir = ROOT
temp_dir.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(
    dir=temp_dir, prefix="P7_", suffix=".h5ad", delete=False
) as handle:
    temp_path = Path(handle.name)

try:
    adata.write_h5ad(temp_path, compression="gzip")
    check = ad.read_h5ad(temp_path)

    assert check.shape == adata.shape
    assert check.obs_names.equals(selected_ids)
    assert check.obs["cell_id_global"].astype(str).tolist() == selected_ids.tolist()
    assert check.obs["cell_type"].astype(str).tolist() == labels.tolist()
    assert check.obs["scanvi_label"].astype(str).tolist() == labels.tolist()
    assert check.obs["sample_id"].isna().all()
    assert check.obs["predicted_doublet"].isna().all()
    assert check.obs["doublet_qc_pass"].isna().all()
    assert check.var["canonical_gene_id"].isna().all()
    assert "tdtomato2" not in check.var_names
    assert check.raw is None
    assert len(check.obsm) == len(check.obsp) == 0
    assert len(check.varm) == len(check.varp) == 0
    assert sparse.isspmatrix_csr(check.X)
    assert sparse.isspmatrix_csr(check.layers["counts"])
    assert (check.X != check.layers["counts"]).nnz == 0
    assert ids_sha256(check.obs_names) == cell_hash
    assert ids_sha256(check.var_names) == feature_hash
    assert csr_sha256(check.X) == x_hash
    assert csr_sha256(check.layers["counts"]) == counts_hash
    assert int(check.layers["counts"].sum()) == previous_total
    assert int(check.layers["counts"].nnz) == previous_nnz

    os.replace(temp_path, output_path)
    if old_path != output_path and old_path.exists():
        old_path.unlink()
finally:
    temp_path.unlink(missing_ok=True)
    try:
        temp_dir.rmdir()
    except OSError:
        pass

print(output_path)
print(f"{adata.n_obs} cells × {adata.n_vars} genes")
print(label_counts)
print("counts: verified UMI; read-back validation: passed")